# Deliver the budget report

*Data Analysis in Python · Topic 16.3*

Your CMO asked which channel groups should get more of next quarter's budget, and how sure you are. Package your analysis so it reruns for any period, put the answer in code, check it against each year on its own, and deliver it as a chart and a written recommendation.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the clean `sessions` table (lesson 3.'s steps) and the cell that builds the `channel_days` table (topic 16.1.'s steps).
2. In the first cell marked ✏️, write `budget_report(channel_days, start, end)`: a function with a docstring that keeps the `channel_days` table's rows from `start` to `end`, and returns topic 16.2.'s `returns` table for them. Then create `report`: the function's table from `START` to `END`.
3. In the second cell marked ✏️, create:
   - `more_budget`: the channel group where the next dollar brings the most
   - `less_budget`: the channel group where it brings the least
   - `confident`: `True` if `more_budget`'s interval starts above the end of `less_budget`'s, and `False` if they overlap
4. In the third cell marked ✏️, create `report_2024` and `report_2025`: the function's tables for each year on its own.
5. In the fourth cell marked ✏️, create `report_ax`: a horizontal bar chart of `report`'s marginal returns, with each one's interval as an error bar, the highest on top, and a headline title.
6. Choose **Insert → Text cell**, and write your recommendation to the CMO under the chart.
7. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `budget_report(channel_days, START, END)` returns topic 16.2.'s table. `more_budget` is `"Paid Search"`, `less_budget` is `"Paid Social"` and `confident` is `True`. In `report_2024` and `report_2025`, Paid Search's marginal return is higher than Paid Social's. `report_ax` has four bars with their intervals.

## Parameters

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
# Every month of 2024 and 2025 as text, "2024-01" to "2025-12": :02d writes the month with two digits
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]
START, END = "2024-01-01", "2025-12-31"
# The media plan's budget lines, by the third part of a campaign's name: Facebook and Instagram share one
CHANNEL_GROUPS = {"search": "Paid Search", "display": "Display", "facebook": "Paid Social", "instagram": "Paid Social", "youtube": "Paid Video"}

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Spend, revenue and organic sessions per channel group and day, with topic 16.1's steps. Run it, and leave it as it is.
def campaign_key(names):
    """Return campaign names stripped of spaces and lowercased, so they match across systems."""
    return names.str.strip().str.lower()


google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts"])
youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])
google["spend"] = google["cost_micros"] / 1_000_000
spend = pd.concat([
    google.rename(columns={"campaign_name": "campaign"})[["date", "campaign", "spend"]],
    meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Amount spent (USD)": "spend"})[["date", "campaign", "spend"]],
    youtube.rename(columns={"Day": "date", "Campaign": "campaign", "Cost": "spend"})[["date", "campaign", "spend"]],
], ignore_index=True)
spend["campaign"] = campaign_key(spend["campaign"])
spend = spend.groupby(["date", "campaign"], as_index=False)["spend"].sum()

revenue = sessions.groupby(["date", "session_campaign"], as_index=False)["revenue"].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = campaign_key(revenue["campaign"])
campaign_days = spend.merge(revenue, on=["date", "campaign"], how="left", validate="one_to_one")
campaign_days["channel_group"] = campaign_days["campaign"].str.split("_").str[2].map(CHANNEL_GROUPS)

organic = sessions[sessions["session_medium"] == "organic"].groupby("date", as_index=False)["sessions"].sum()
organic = organic.rename(columns={"sessions": "organic"})
channel_days = campaign_days.groupby(["channel_group", "date"], as_index=False)[["spend", "revenue"]].sum()
channel_days = channel_days.merge(organic, on="date", how="left", validate="many_to_one")

checks = {
    "every campaign is in a channel group": bool(campaign_days["channel_group"].notna().all()),
    "every campaign day has a revenue value": bool(campaign_days["revenue"].notna().all()),
    "every channel group has every day": len(channel_days) == len(set(CHANNEL_GROUPS.values())) * len(pd.date_range(START, END)),
}
assert all(checks.values()), checks
channel_days.shape

## The report, for any period

In [ ]:
# ✏️ Write budget_report(channel_days, start, end), and create report


## The answer

In [ ]:
# ✏️ Create more_budget, less_budget and confident


## Each year on its own

In [ ]:
# ✏️ Create report_2024 and report_2025


## The chart

In [ ]:
# ✏️ Create report_ax


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell reads your variables, your chart and the cells that have run, and calls your budget_report() with a period of its own. It doesn't change anything.
def check_your_work():
    import sys
    import matplotlib
    import pandas as pd
    from matplotlib.container import ErrorbarContainer
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    groups = {"Paid Search", "Paid Video", "Display", "Paid Social"}
    columns = {"roas", "marginal_return", "return_low", "return_high"}

    def table(value):
        return isinstance(value, pd.DataFrame) and columns <= set(value.columns) and set(value.index) == groups

    def roas_matches(value, expected):
        return table(value) and all(abs(value.loc[group, "roas"] - roas) < 0.0001 for group, roas in expected.items())

    # Each channel group's ROAS over the full weeks of a period: what topic 16.2.'s steps give
    full = {"Paid Search": 2.381396, "Paid Video": 1.618471, "Display": 1.604192, "Paid Social": 2.021792}
    probe = {"Paid Search": 2.340158, "Paid Video": 1.703729, "Display": 1.704698, "Paid Social": 2.081728}
    year_2024 = {"Paid Search": 2.226595, "Paid Video": 1.476318, "Display": 1.654630, "Paid Social": 1.985369}
    year_2025 = {"Paid Search": 2.504620, "Paid Video": 1.732859, "Display": 1.558516, "Paid Social": 2.041344}

    budget_report, channel_days = found.get("budget_report"), found.get("channel_days")
    if not callable(budget_report):
        results.append((False, "budget_report should be a function: def budget_report(channel_days, start, end): ..."))
    elif not isinstance(channel_days, pd.DataFrame) or len(channel_days) != 2924:
        results.append((False, "Run the cell that builds the channel_days table, and leave it as it is."))
    else:
        before = channel_days.copy()
        try:
            probed = budget_report(channel_days, "2024-07-01", "2025-06-30")
            problem = None
        except Exception as error:
            probed, problem = None, f"{type(error).__name__}: {error}"
        if problem:
            results.append((False, f"budget_report(channel_days, \"2024-07-01\", \"2025-06-30\") stops with {problem}"))
        elif not channel_days.equals(before):
            results.append((False, "budget_report() changes the channel_days table it's given. Work on the rows you keep, and leave the table as it is."))
        elif not table(probed):
            results.append((False, "budget_report() should return a DataFrame with the four channel groups as its index, and the columns roas, marginal_return, return_low and return_high."))
        elif roas_matches(probed, full):
            results.append((False, "budget_report(channel_days, \"2024-07-01\", \"2025-06-30\") returns the table for 2024 and 2025: the function reads START and END instead of start and end. Keep the rows from start to end."))
        elif not roas_matches(probed, probe):
            results.append((False, f"From July 2024 to June 2025, budget_report() gives Paid Search a ROAS of {probed.loc['Paid Search', 'roas']:.4f}. Topic 16.2.'s steps on those days give 2.3402: keep the rows whose date is between start and end, then the full weeks."))
        elif list(probed.index) != list(probed.sort_values("marginal_return", ascending=False).index):
            results.append((False, "budget_report() should sort its table by marginal_return, highest first."))
        else:
            results.append((True, f"budget_report() works for a period of the check's choosing: from July 2024 to June 2025, the next dollar brings the most in {probed.index[0]}, ${probed.iloc[0]['marginal_return']:.2f}."))

    report = found.get("report")
    if not table(report):
        results.append((False, "report should be budget_report(channel_days, START, END): a DataFrame with the four channel groups as its index."))
    elif not roas_matches(report, full):
        results.append((False, f"report should be the table for START to END, topic 16.2.'s: Paid Search's ROAS 2.3814. It's {report.loc['Paid Search', 'roas']:.4f}."))
    elif report.loc["Paid Search", "marginal_return"] > report.loc["Paid Search", "roas"]:
        results.append((False, "In report, Paid Search's marginal return is more than its ROAS: the model has no control for demand. Use \"revenue ~ spend + organic\", as in topic 16.2."))
    else:
        results.append((True, f"report is topic 16.2.'s table: from {report.index[0]}'s ${report.iloc[0]['marginal_return']:.2f} down to {report.index[-1]}'s ${report.iloc[-1]['marginal_return']:.2f}."))

    more_budget, less_budget, confident = found.get("more_budget"), found.get("less_budget"), found.get("confident")
    if not isinstance(more_budget, str) or not isinstance(less_budget, str):
        results.append((False, "more_budget and less_budget should be the names of channel groups: report.index[0] and report.index[-1]."))
    elif more_budget == "Paid Social" and less_budget == "Paid Search":
        results.append((False, "more_budget and less_budget are the wrong way round: the report is sorted with the highest marginal return first, so more_budget is report.index[0]."))
    elif less_budget == "Display":
        results.append((False, "less_budget is Display, the channel group with the lowest ROAS. The question is about the next dollar: take the lowest marginal return, report.index[-1]."))
    elif more_budget != "Paid Search" or less_budget != "Paid Social":
        results.append((False, f"more_budget should be the first row of report, and less_budget the last: report.index[0] and report.index[-1]. They are {more_budget!r} and {less_budget!r}."))
    elif not isinstance(confident, bool):
        results.append((False, f"confident should be True or False: bool(report.loc[more_budget, \"return_low\"] > report.loc[less_budget, \"return_high\"]). It is {confident!r}."))
    elif confident is not True:
        results.append((False, "confident is False, but Paid Search's interval starts at 1.60, above the end of Paid Social's at 1.28. Compare more_budget's return_low with less_budget's return_high."))
    else:
        results.append((True, "Move budget from Paid Social to Paid Search, and be confident about it: their intervals don't overlap."))

    report_2024, report_2025 = found.get("report_2024"), found.get("report_2025")
    if not table(report_2024) or not table(report_2025):
        results.append((False, "report_2024 and report_2025 should be budget_report()'s tables for each year: from \"2024-01-01\" to \"2024-12-31\", and from \"2025-01-01\" to \"2025-12-31\"."))
    elif not roas_matches(report_2024, year_2024) or not roas_matches(report_2025, year_2025):
        results.append((False, f"report_2024 and report_2025 should cover one year each. Their Paid Search ROAS is {report_2024.loc['Paid Search', 'roas']:.4f} and {report_2025.loc['Paid Search', 'roas']:.4f}, and each year on its own gives 2.2266 and 2.5046."))
    else:
        both = all(year.loc["Paid Search", "marginal_return"] > year.loc["Paid Social", "marginal_return"] for year in (report_2024, report_2025))
        results.append((True, f"Each year on its own: Paid Search ${report_2024.loc['Paid Search', 'marginal_return']:.2f} and ${report_2025.loc['Paid Search', 'marginal_return']:.2f}, Paid Social ${report_2024.loc['Paid Social', 'marginal_return']:.2f} and ${report_2025.loc['Paid Social', 'marginal_return']:.2f}. {'Both years point the same way.' if both else 'The years disagree.'}"))

    ax = found.get("report_ax")
    if not isinstance(ax, matplotlib.axes.Axes):
        results.append((False, f"report_ax should be the Axes of your bar chart: report_ax = chart[\"marginal_return\"].plot.barh(xerr=errors). It is a {type(ax).__name__}."))
    elif table(report):
        bars = [patch for patch in ax.patches if patch.get_width() != 0 or patch.get_height() != 0]
        widths = sorted(round(float(bar.get_width()), 4) for bar in bars)
        intervals = [container for container in ax.containers if isinstance(container, ErrorbarContainer)]
        segments = [segment for container in intervals for collection in container.lines[2] for segment in collection.get_segments()]
        ends = sorted((round(float(min(segment[:, 0])), 4), round(float(max(segment[:, 0])), 4)) for segment in segments)
        expected_ends = sorted((round(float(low), 4), round(float(high), 4)) for low, high in zip(report["return_low"], report["return_high"]))
        if widths != sorted(round(float(value), 4) for value in report["marginal_return"]):
            results.append((False, f"report_ax should have one horizontal bar per channel group, as long as its marginal return. It has {len(bars)} bars: chart[\"marginal_return\"].plot.barh()."))
        elif ends != expected_ends:
            results.append((False, "Each bar should have its interval as an error bar, from return_low to return_high. Calculate errors from the same sorted table you plot, so each interval stays with its bar: xerr=[chart[\"marginal_return\"] - chart[\"return_low\"], chart[\"return_high\"] - chart[\"marginal_return\"]]."))
        elif round(float((min if ax.yaxis_inverted() else max)(bars, key=lambda bar: bar.get_y()).get_width()), 4) != round(float(report["marginal_return"].max()), 4):
            results.append((False, "Put the highest marginal return on top: sort the table by marginal_return before plot.barh(), as in topic 8.1."))
        elif not ax.get_title():
            results.append((False, "The chart has no title. State the finding, as in topic 8.4.: report_ax.set_title(\"...\")."))
        else:
            results.append((True, f"The chart shows each channel group's next dollar with its interval: \"{ax.get_title()}\"."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Read your recommendation once more, as your CMO will: the check can't read it for you.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
def budget_report(channel_days, start, end):
    """Return each channel group's ROAS and marginal return, with its interval, from start to end."""
    period = channel_days[channel_days["date"].between(start, end)]
    rows = []
    for group, group_days in period.groupby("channel_group"):
        group_days = group_days.set_index("date")
        days = group_days["spend"].resample("W").count()
        weekly = group_days[["spend", "revenue", "organic"]].resample("W").sum()
        weekly = weekly[days == 7]
        model = smf.ols("revenue ~ spend + organic", data=weekly).fit()
        low, high = model.conf_int().loc["spend"]
        rows.append({"channel_group": group, "roas": weekly["revenue"].sum() / weekly["spend"].sum(),
                     "marginal_return": model.params["spend"], "return_low": low, "return_high": high})
    return pd.DataFrame(rows).set_index("channel_group").sort_values("marginal_return", ascending=False)


report = budget_report(channel_days, START, END)
report
```

```python
more_budget, less_budget = report.index[0], report.index[-1]
confident = bool(report.loc[more_budget, "return_low"] > report.loc[less_budget, "return_high"])
more_budget, less_budget, confident
```

```python
report_2024 = budget_report(channel_days, "2024-01-01", "2024-12-31")
report_2025 = budget_report(channel_days, "2025-01-01", "2025-12-31")
report_2024["marginal_return"], report_2025["marginal_return"]
```

```python
chart = report.sort_values("marginal_return")
errors = [chart["marginal_return"] - chart["return_low"], chart["return_high"] - chart["marginal_return"]]
report_ax = chart["marginal_return"].plot.barh(xerr=errors)
report_ax.set_xlabel("Revenue from one more dollar of spend, 2024–2025, with 95% intervals")
report_ax.set_title("Paid Search's next dollar brings the most, Paid Social's the least")
plt.show()
```

```markdown
**Move part of Paid Social's budget to Paid Search next quarter.** Paid Search's next dollar brings about $1.98 of revenue (95% interval $1.60 to $2.37), and Paid Social's about $0.55 ($−0.17 to $1.28). The intervals don't overlap, so we can be confident the move brings more revenue for the same budget. Paid Social's ROAS of $2.02 hides this: it's the average of every dollar, and its cheapest clicks are already bought.

Each year on its own points the same way: Paid Search $1.81 in 2024 and $2.37 in 2025, Paid Social $0.53 and $0.57. Paid Video and Display sit in between, and the data can't tell them apart from either end, so keep their budgets.

The model only knows the weekly spend we've had, so move part of the budget, not all of it, and rerun this report when Q1 2026's data is in to see whether Paid Search's return holds.
```

</details>